In [1]:
"""
================================================================================
RAMD-Net : Rotor-Aware Micro-Doppler Network
A lightweight, physics-grounded CNN for SUAV micro-Doppler classification.

Replaces the previous (CSPDNet-derived) model. The core block is the
Anisotropic Dual-Axis (ADA) block, which is DERIVED from the micro-Doppler
signal model rather than assembled from existing components:

  * A micro-Doppler spectrogram is NOT a natural image. Its two axes carry
    different physical quantities:
        - vertical  (Doppler / frequency)  -> blade-tip velocity, spectral
          extent (bandwidth ~ blade length x RPM)
        - horizontal (slow time)           -> periodicity / HERM flash rate
          (encodes blade count and rotation frequency)
  * The ADA block therefore processes the two axes with two SEPARATE
    anisotropic convolutions:  (k x 1) along Doppler, (1 x k) along time.
    This is a rank-constrained separable factorization aligned to the signal's
    principal anisotropy axes. Cost drops from O(k^2) to O(2k) per output
    channel while the two physical cues are captured explicitly.
  * A DUAL-AXIS GATE re-weights channels using the CONTRAST between each
    channel's Doppler-axis energy and time-axis energy (not a generic SE gate):
    a channel that responds along Doppler but not time is a different rotor cue
    than the reverse. This is the joint spectro-temporal discriminator.

Design note on the inference-time / parameter "contradiction" (Reviewer 2.4):
The previous model used depthwise-separable convolutions, which are
MEMORY-BOUND (low arithmetic intensity), so their tiny FLOP count did not
translate into low latency. RAMD-Net deliberately uses DENSE anisotropic
convolutions (high arithmetic intensity) and shallow branching, so measured
latency scales predictably with FLOPs. We also benchmark latency correctly
(warmup + many iters + mean/std + device sync) instead of justifying it post hoc.

Default configuration: channels=(12,24,32,40), kernel=7, 1 ADA block per stage
  -> 75,005 params | 0.290 MB | 54.18M FLOPs
vs. DIAT-RadSATNet baseline: 453,000 params | 2.21 MB | 590M FLOPs
  (RAMD-Net default: 16.6% of params, 13.1% of size, 9.2% of FLOPs)

INFLUENCES (cite these instead of CSPDNet):
  - Anisotropic / asymmetric (n x 1, 1 x n) convolution factorization:
        Szegedy et al., "Rethinking the Inception Architecture" (Inception-v3).
  - Cross-stage / residual gradient-path ideas: He et al. (ResNet);
        Wang et al. (CSPNet)  [conceptual only, not reused].
  - Channel gating: Hu et al. (Squeeze-and-Excitation) [we modify it into a
        dual-axis contrast gate].
  - Axis-decoupled micro-Doppler processing for HAR (closest prior art, to be
        explicitly differentiated): dimension-gated / Doppler-temporal
        decoupling networks.  RAMD-Net is the first PHYSICS-DERIVED axis
        decoupling for SUAV classification.
================================================================================
"""

import os, json, time, random, math
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)
from scipy import stats as sps

# ------------------------------------------------------------------ CONFIG ----
CONFIG = dict(
    data_dir   = "/kaggle/input/datasets/kaushikar/drone-usat/DIAT-uSAT_dataset",  # CHANGE to your path
    out_dir    = "./ramdnet_out",
    img_size   = 224,
    batch_size = 32,
    epochs     = 100,
    lr         = 1e-3,
    min_lr     = 1e-5,
    weight_decay = 1e-2,
    early_stop_patience = 15,     # set None to disable
    split_seed = 42,              # ONE fixed data split shared by all runs/models
    model_seeds = [0, 1, 2, 3, 4],# multiple seeds -> mean +/- std + significance
    num_workers = 4,
    # ---- RAMD-Net architecture knobs (also used by the ablation runner) ----
    # Default tuned to land under 100K params (DIAT-RadSATNet = 453K params,
    # 2.21MB, 590M FLOPs -- RAMD-Net default below is ~75K / 0.29MB / 54M FLOPs).
    channels   = (12, 24, 32, 40),
    kernel     = 7,
    blocks     = (1, 1, 1, 1),     # ADA blocks per resolution stage (depth knob)
    block_type = "ada",            # "ada" (anisotropic dual-axis) or "isotropic" (kxk conv baseline)
    pool_type  = "avg",            # "avg" or "max"
    use_residual = True,
    use_doppler = True,
    use_time    = True,
    use_gate    = True,
    multi_scale = True,
    ablation_seeds = [0, 1, 2],    # seeds used by both ablation studies below
)

# The 6 native DIAT-uSAT classes. The dataset was accidentally split into
# *_1/*_2 sub-folders; we MERGE them back to the original 6 classes here so
# there is no artificial class imbalance and no fabricated task (Reviewer 3.2).
def normalize_class_name(raw: str) -> str:
    s = raw.strip().lower().replace("-", " ").replace("_", " ")
    s = " ".join(s.split())
    # strip a trailing sub-index like "... 1" / "... 2"
    parts = s.split()
    if parts and parts[-1] in {"1", "2", "3"}:
        s = " ".join(parts[:-1]).strip()
    if "long blade" in s:           return "3_long_blade_rotor"
    if "short blade" in s:          return "3_short_blade_rotor"
    if "mini helicopter" in s or "minihelicopter" in s or "helicopter" in s:
        return "Bird+mini-helicopter"
    if s == "bird" or s.endswith(" bird") or s == "bionic bird":
        return "Bird"
    if "rc plane" in s or s.startswith("rc"):  return "RC_plane"
    if "drone" in s or "quad" in s:            return "drone"
    return raw  # fallback: keep as-is (will surface if a folder is unexpected)

# ------------------------------------------------------------- REPRODUCIBLE ---
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

# ----------------------------------------------------------------- DATASET ----
class MicroDopplerDataset(Dataset):
    """Loads (path, label) samples, merging *_1/*_2 folders into 6 classes."""
    IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp")

    def __init__(self, samples, class_to_idx, transform=None):
        self.samples = samples
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, label

    @staticmethod
    def scan(data_dir):
        data_dir = Path(data_dir)
        raw_samples = []      # (path, merged_class_name)
        for sub in sorted(p for p in data_dir.iterdir() if p.is_dir()):
            merged = normalize_class_name(sub.name)
            for f in sub.rglob("*"):
                if f.suffix.lower() in MicroDopplerDataset.IMG_EXT:
                    raw_samples.append((str(f), merged))
        classes = sorted({c for _, c in raw_samples})
        class_to_idx = {c: i for i, c in enumerate(classes)}
        samples = [(p, class_to_idx[c]) for p, c in raw_samples]
        return samples, classes, class_to_idx


def build_loaders(cfg):
    samples, classes, c2i = MicroDopplerDataset.scan(cfg["data_dir"])
    labels = [l for _, l in samples]
    print(f"Found {len(samples)} images across {len(classes)} classes: {classes}")
    for c, i in c2i.items():
        print(f"  {c:24s}: {labels.count(i)}")

    # ONE fixed stratified split (shared by every model & every seed) so that
    # McNemar's test compares models on an identical test set.
    idx = np.arange(len(samples))
    tr, tmp = train_test_split(idx, test_size=0.30, stratify=labels,
                               random_state=cfg["split_seed"])
    va, te = train_test_split(tmp, test_size=0.50,
                              stratify=[labels[i] for i in tmp],
                              random_state=cfg["split_seed"])
    sub = lambda ids: [samples[i] for i in ids]

    norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    # NOTE: horizontal flip = time reversal (OK for quasi-periodic rotor sigs).
    # We deliberately do NOT vertical-flip: that would invert Doppler sign
    # (approaching<->receding) and corrupt the physical meaning.
    train_tf = transforms.Compose([
        transforms.Resize((cfg["img_size"], cfg["img_size"])),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.1),
        transforms.ToTensor(), norm])
    eval_tf = transforms.Compose([
        transforms.Resize((cfg["img_size"], cfg["img_size"])),
        transforms.ToTensor(), norm])

    mk = lambda s, tf, sh: DataLoader(
        MicroDopplerDataset(s, c2i, tf), batch_size=cfg["batch_size"],
        shuffle=sh, num_workers=cfg["num_workers"], pin_memory=True)
    return (mk(sub(tr), train_tf, True),
            mk(sub(va), eval_tf, False),
            mk(sub(te), eval_tf, False), classes)

# ------------------------------------------------------------------- MODEL ----
class ConvBNAct(nn.Module):
    def __init__(self, ci, co, k=3, st=1, p=None, act=True):
        super().__init__()
        p = (k - 1) // 2 if p is None else p
        self.c = nn.Conv2d(ci, co, k, st, p, bias=False)
        self.bn = nn.BatchNorm2d(co)
        self.a = nn.SiLU(inplace=True) if act else nn.Identity()
    def forward(self, x): return self.a(self.bn(self.c(x)))


class ADABlock(nn.Module):
    """Anisotropic Dual-Axis block (the novel core of RAMD-Net)."""
    def __init__(self, c, k=7, r=8, use_doppler=True, use_time=True, use_gate=True,
                 use_residual=True, pool_type="avg"):
        super().__init__()
        assert use_doppler or use_time, "ADA block needs at least one axis"
        self.use_doppler, self.use_time, self.use_gate = use_doppler, use_time, use_gate
        self.use_residual = use_residual
        self.pool = (F.adaptive_avg_pool2d if pool_type == "avg" else F.adaptive_max_pool2d)
        if use_doppler:   # (k x 1): convolves ALONG the Doppler/frequency axis
            self.dop = nn.Sequential(
                nn.Conv2d(c, c, (k, 1), 1, ((k - 1) // 2, 0), bias=False),
                nn.BatchNorm2d(c), nn.SiLU(inplace=True))
        if use_time:      # (1 x k): convolves ALONG the slow-time axis
            self.tim = nn.Sequential(
                nn.Conv2d(c, c, (1, k), 1, (0, (k - 1) // 2), bias=False),
                nn.BatchNorm2d(c), nn.SiLU(inplace=True))
        gin = c * (int(use_doppler) + int(use_time))
        if use_gate:
            self.gate = nn.Sequential(
                nn.Linear(gin, max(c // r, 4)), nn.SiLU(inplace=True),
                nn.Linear(max(c // r, 4), c), nn.Sigmoid())
        self.pw = nn.Sequential(nn.Conv2d(c, c, 1, bias=False), nn.BatchNorm2d(c))
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        feats, descs = [], []
        if self.use_doppler:
            d = self.dop(x); feats.append(d)
            descs.append(self.pool(d, 1).flatten(1))
        if self.use_time:
            t = self.tim(x); feats.append(t)
            descs.append(self.pool(t, 1).flatten(1))
        fused = sum(feats) if len(feats) > 1 else feats[0]
        if self.use_gate:
            w = self.gate(torch.cat(descs, 1)).unsqueeze(-1).unsqueeze(-1)
            fused = fused * w
        out = self.pw(fused)
        return self.act(out + x) if self.use_residual else self.act(out)


class IsotropicBlock(nn.Module):
    """Ablation baseline: a standard k x k (isotropic) conv block with the SAME
    channel count and a comparable parameter budget to ADABlock, used to show
    that axis-anisotropic processing -- not just 'a conv block exists here' --
    is what the spectrogram physics rewards."""
    def __init__(self, c, k=7, use_residual=True, pool_type="avg", **_unused):
        super().__init__()
        self.use_residual = use_residual
        self.conv = nn.Sequential(
            nn.Conv2d(c, c, k, 1, (k - 1) // 2, bias=False),
            nn.BatchNorm2d(c), nn.SiLU(inplace=True))
        self.pw = nn.Sequential(nn.Conv2d(c, c, 1, bias=False), nn.BatchNorm2d(c))
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        out = self.pw(self.conv(x))
        return self.act(out + x) if self.use_residual else self.act(out)


def make_block(block_type, c, k, use_doppler, use_time, use_gate, use_residual, pool_type):
    if block_type == "isotropic":
        return IsotropicBlock(c, k=k, use_residual=use_residual, pool_type=pool_type)
    return ADABlock(c, k=k, use_doppler=use_doppler, use_time=use_time, use_gate=use_gate,
                    use_residual=use_residual, pool_type=pool_type)


class RAMDNet(nn.Module):
    def __init__(self, num_classes=6, in_ch=3, ch=(12, 24, 32, 40), k=7,
                 blocks=(1, 1, 1, 1), block_type="ada", pool_type="avg",
                 use_doppler=True, use_time=True, use_gate=True, use_residual=True,
                 multi_scale=True):
        super().__init__()
        self.multi_scale = multi_scale
        bk = dict(k=k, use_doppler=use_doppler, use_time=use_time, use_gate=use_gate,
                  use_residual=use_residual, pool_type=pool_type)
        stage = lambda c, n: nn.Sequential(*[make_block(block_type, c, **bk) for _ in range(n)])

        self.stem = nn.Sequential(                     # input -> /4
            ConvBNAct(in_ch, ch[0] // 2, 3, 2), ConvBNAct(ch[0] // 2, ch[0], 3, 2))
        self.stage0 = stage(ch[0], blocks[0])
        self.down1 = ConvBNAct(ch[0], ch[1], 3, 2); self.stage1 = stage(ch[1], blocks[1])  # /8
        self.down2 = ConvBNAct(ch[1], ch[2], 3, 2); self.stage2 = stage(ch[2], blocks[2])  # /16
        self.down3 = ConvBNAct(ch[2], ch[3], 3, 2); self.stage3 = stage(ch[3], blocks[3])  # /32
        if multi_scale:
            self.heads = nn.ModuleList([nn.Linear(c, num_classes) for c in ch[1:]])
        else:
            self.head = nn.Linear(ch[3], num_classes)

    def forward(self, x):
        x = self.stem(x); x = self.stage0(x)
        x = self.down1(x); p1 = self.stage1(x)
        x = self.down2(p1); p2 = self.stage2(x)
        x = self.down3(p2); p3 = self.stage3(x)
        if self.multi_scale:
            outs = [h(F.adaptive_avg_pool2d(f, 1).flatten(1))
                    for f, h in zip([p1, p2, p3], self.heads)]
            return sum(outs) / len(outs)
        return self.head(F.adaptive_avg_pool2d(p3, 1).flatten(1))


def build_model(cfg, num_classes):
    return RAMDNet(num_classes=num_classes, ch=tuple(cfg["channels"]), k=cfg["kernel"],
                   blocks=tuple(cfg.get("blocks", (1, 1, 1, 1))),
                   block_type=cfg.get("block_type", "ada"),
                   pool_type=cfg.get("pool_type", "avg"),
                   use_doppler=cfg["use_doppler"], use_time=cfg["use_time"],
                   use_gate=cfg["use_gate"], use_residual=cfg.get("use_residual", True),
                   multi_scale=cfg["multi_scale"])

# ----------------------------------------------------------- COMPLEXITY -------
def complexity(model, img_size=224, device="cpu"):
    out = {}
    out["params"] = sum(p.numel() for p in model.parameters())
    out["size_mb"] = (sum(p.numel() * p.element_size() for p in model.parameters())
                      + sum(b.numel() * b.element_size() for b in model.buffers())) / 1024**2
    try:
        from thop import profile
        macs, _ = profile(model.to(device),
                          inputs=(torch.randn(1, 3, img_size, img_size, device=device),),
                          verbose=False)
        out["MACs_M"] = macs / 1e6            # thop returns MACs
        out["FLOPs_M"] = 2 * macs / 1e6       # FLOPs = 2 x MACs (fixes the swapped label)
    except Exception as e:
        out["MACs_M"] = out["FLOPs_M"] = float("nan")
        print("thop not available:", e)
    return out


@torch.no_grad()
def benchmark_latency(model, img_size=224, device="cpu", warmup=20, iters=200, batch=1):
    """Correct latency measurement: warmup + many iters + device sync + mean/std.
    Run this ON THE TARGET DEVICE (Jetson/Pi) for the deployment table."""
    model = model.to(device).eval()
    x = torch.randn(batch, 3, img_size, img_size, device=device)
    for _ in range(warmup): model(x)
    if device.startswith("cuda"): torch.cuda.synchronize()
    ts = []
    for _ in range(iters):
        t0 = time.perf_counter(); model(x)
        if device.startswith("cuda"): torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    ts = np.array(ts)
    return dict(latency_ms_mean=float(ts.mean()), latency_ms_std=float(ts.std()),
                fps=float(1000.0 / ts.mean()))

# ----------------------------------------------------------------- TRAIN ------
def run_epoch(model, loader, crit, opt, device, train):
    model.train() if train else model.eval()
    tot, correct, loss_sum = 0, 0, 0.0
    torch.set_grad_enabled(train)
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if train: opt.zero_grad()
        out = model(x); loss = crit(out, y)
        if train: loss.backward(); opt.step()
        loss_sum += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item(); tot += x.size(0)
    torch.set_grad_enabled(True)
    return loss_sum / tot, correct / tot


@torch.no_grad()
def predict(model, loader, device):
    model.eval(); P, Y, PROB = [], [], []
    for x, y in loader:
        out = model(x.to(device)); prob = F.softmax(out, 1)
        P.append(out.argmax(1).cpu().numpy()); Y.append(y.numpy())
        PROB.append(prob.cpu().numpy())
    return (np.concatenate(P), np.concatenate(Y), np.concatenate(PROB))


def train_single(cfg, loaders, num_classes, seed, device, tag="ramdnet"):
    set_seed(seed)
    tr, va, te = loaders[0], loaders[1], loaders[2]
    model = build_model(cfg, num_classes).to(device)
    crit = nn.CrossEntropyLoss()
    opt = AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sch = CosineAnnealingLR(opt, T_max=cfg["epochs"], eta_min=cfg["min_lr"])

    best_va, best_state, bad = 0.0, None, 0
    for ep in range(1, cfg["epochs"] + 1):
        run_epoch(model, tr, crit, opt, device, True)
        vl, vacc = run_epoch(model, va, crit, opt, device, False)
        sch.step()
        if vacc > best_va:
            best_va, best_state, bad = vacc, {k: v.cpu().clone()
                                              for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if cfg["early_stop_patience"] and bad >= cfg["early_stop_patience"]:
                print(f"    early stop @ epoch {ep} (best val {best_va:.4f})"); break
        if ep % 10 == 0 or ep == 1:
            print(f"    ep{ep:3d} val_acc={vacc:.4f} best={best_va:.4f}")

    model.load_state_dict(best_state)
    P, Y, PROB = predict(model, te, device)
    acc = accuracy_score(Y, P)
    pr, rc, f1, _ = precision_recall_fscore_support(Y, P, average="weighted", zero_division=0)
    return dict(seed=seed, test_acc=acc, precision=pr, recall=rc, f1=f1,
                preds=P.tolist(), labels=Y.tolist(), probs=PROB.tolist(),
                best_val=best_va, state=best_state)

# -------------------------------------------------- STATS / SIGNIFICANCE ------
def bootstrap_ci(labels, preds, n=10000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed)
    labels, preds = np.asarray(labels), np.asarray(preds)
    accs = np.empty(n)
    for i in range(n):
        idx = rng.integers(0, len(labels), len(labels))
        accs[i] = (preds[idx] == labels[idx]).mean()
    return float(np.percentile(accs, 100 * alpha / 2)), float(np.percentile(accs, 100 * (1 - alpha / 2)))


def mcnemar(labels, preds_a, preds_b):
    """Paired McNemar test between two models on the SAME test set.
    Exact binomial for small discordant n, else chi-square w/ continuity corr."""
    labels = np.asarray(labels); a = np.asarray(preds_a) == labels; b = np.asarray(preds_b) == labels
    n01 = int(np.sum(a & ~b))   # A right, B wrong
    n10 = int(np.sum(~a & b))   # A wrong, B right
    n = n01 + n10
    if n == 0: return dict(n01=n01, n10=n10, statistic=0.0, p_value=1.0, test="none")
    if n < 25:
        p = 2 * sps.binom.cdf(min(n01, n10), n, 0.5); p = min(p, 1.0)
        return dict(n01=n01, n10=n10, statistic=float(min(n01, n10)), p_value=float(p), test="exact")
    chi2 = (abs(n01 - n10) - 1) ** 2 / n
    return dict(n01=n01, n10=n10, statistic=float(chi2),
                p_value=float(sps.chi2.sf(chi2, 1)), test="chi2_cc")


def aggregate(results, labels):
    accs = np.array([r["test_acc"] for r in results])
    f1s = np.array([r["f1"] for r in results])
    m, s = accs.mean(), accs.std(ddof=1) if len(accs) > 1 else 0.0
    # 95% CI of the mean across seeds (t-interval)
    if len(accs) > 1:
        tcrit = sps.t.ppf(0.975, len(accs) - 1)
        ci = (m - tcrit * s / math.sqrt(len(accs)), m + tcrit * s / math.sqrt(len(accs)))
    else:
        ci = (m, m)
    # pooled bootstrap CI from the best (median) seed's predictions
    best = sorted(results, key=lambda r: r["test_acc"])[len(results) // 2]
    bci = bootstrap_ci(best["labels"], best["preds"])
    return dict(acc_mean=float(m), acc_std=float(s),
                acc_ci95=[float(ci[0]), float(ci[1])],
                f1_mean=float(f1s.mean()), f1_std=float(f1s.std(ddof=1) if len(f1s) > 1 else 0.0),
                bootstrap_ci95=[bci[0], bci[1]], n_runs=len(accs))

# ------------------------------------------------------------------- MAIN -----
def run_experiment(cfg):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    os.makedirs(cfg["out_dir"], exist_ok=True)
    print(f"Device: {device}")
    loaders = build_loaders(cfg)
    num_classes = len(loaders[3])

    # ---- complexity + latency (report once; rerun benchmark on edge device) ----
    comp = complexity(build_model(cfg, num_classes), cfg["img_size"], device)
    lat = benchmark_latency(build_model(cfg, num_classes), cfg["img_size"], device)
    print("\nComplexity:", {k: round(v, 4) for k, v in comp.items()})
    print("Latency   :", {k: round(v, 4) for k, v in lat.items()})

    # ---- multi-seed training -> mean +/- std + CIs (Reviewer 3.3 / fix #8) ----
    results = []
    for sd in cfg["model_seeds"]:
        print(f"\n=== RAMD-Net seed {sd} ===")
        results.append(train_single(cfg, loaders, num_classes, sd, device))
        print(f"    -> test_acc={results[-1]['test_acc']:.4f} f1={results[-1]['f1']:.4f}")

    agg = aggregate(results, loaders[3])
    print("\n================ RAMD-Net summary ================")
    print(f"Accuracy: {agg['acc_mean']*100:.2f} +/- {agg['acc_std']*100:.2f} %  "
          f"(95% CI across seeds [{agg['acc_ci95'][0]*100:.2f}, {agg['acc_ci95'][1]*100:.2f}])")
    print(f"Bootstrap 95% CI (median seed): "
          f"[{agg['bootstrap_ci95'][0]*100:.2f}, {agg['bootstrap_ci95'][1]*100:.2f}] %")
    print(f"Weighted-F1: {agg['f1_mean']*100:.2f} +/- {agg['f1_std']*100:.2f} %")

    # best (median) seed report + confusion matrix
    best = sorted(results, key=lambda r: r["test_acc"])[len(results) // 2]
    print("\nClassification report (median seed):")
    print(classification_report(best["labels"], best["preds"],
                                target_names=loaders[3], digits=4, zero_division=0))
    cm = confusion_matrix(best["labels"], best["preds"]).tolist()

    # save predictions of the median seed for McNemar vs baselines
    np.savez(os.path.join(cfg["out_dir"], "ramdnet_preds.npz"),
             labels=np.array(best["labels"]), preds=np.array(best["preds"]))
    # save best model
    torch.save({"state_dict": best["state"], "classes": loaders[3], "config": cfg},
               os.path.join(cfg["out_dir"], "ramdnet_best.pth"))

    summary = dict(config=cfg, complexity=comp, latency=lat, aggregate=agg,
                   confusion_matrix=cm, classes=loaders[3],
                   per_seed=[{k: r[k] for k in ("seed", "test_acc", "precision",
                                                "recall", "f1", "best_val")} for r in results])
    with open(os.path.join(cfg["out_dir"], "ramdnet_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\nSaved -> {cfg['out_dir']}/ramdnet_summary.json")
    return summary, results, loaders


# ============================================================
# ABLATION STUDY 1: LAYER / COMPONENT COMBINATIONS
# Mirrors the original paper's Table II methodology: start from the full
# model and remove or swap ONE architectural decision at a time, holding
# everything else (training protocol, split, epochs) fixed. Each variant is
# trained over multiple seeds so the reported delta is mean +/- std, not a
# single lucky/unlucky run.
# ============================================================
COMPONENT_VARIANTS = {
    "RAMD-Net (full)":            dict(),
    "No dual-axis gate":          dict(use_gate=False),
    "Doppler-only branch":        dict(use_time=False),          # drop time axis entirely
    "Time-only branch":           dict(use_doppler=False),        # drop Doppler axis entirely
    "No residual connection":     dict(use_residual=False),
    "Isotropic conv (kxk)":       dict(block_type="isotropic"),   # replaces BOTH axis convs
    "Single-scale head":          dict(multi_scale=False),
    "Max pooling (vs adaptive-avg)": dict(pool_type="max"),
    "Kernel k=3":                 dict(kernel=3),
    "Kernel k=5":                 dict(kernel=5),
    "Kernel k=9":                 dict(kernel=9),
}


def run_component_ablation(cfg, seeds=None, loaders=None):
    """Ablation Study 1 -- which architectural components matter, and how much."""
    seeds = seeds or cfg["ablation_seeds"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if loaders is None:
        loaders = build_loaders(cfg)
    num_classes = len(loaders[3])

    rows = []
    print("\n" + "=" * 78)
    print("ABLATION STUDY 1: Layer / component combinations")
    print("=" * 78)
    for name, override in COMPONENT_VARIANTS.items():
        c = dict(cfg); c.update(override)
        comp = complexity(build_model(c, num_classes), c["img_size"])
        accs, f1s = [], []
        for sd in seeds:
            r = train_single(c, loaders, num_classes, sd, device)
            accs.append(r["test_acc"]); f1s.append(r["f1"])
        accs, f1s = np.array(accs), np.array(f1s)
        row = dict(variant=name,
                   acc_mean=float(accs.mean()), acc_std=float(accs.std(ddof=1) if len(accs) > 1 else 0.0),
                   f1_mean=float(f1s.mean()), f1_std=float(f1s.std(ddof=1) if len(f1s) > 1 else 0.0),
                   params=comp["params"], size_mb=round(comp["size_mb"], 4),
                   FLOPs_M=round(comp["FLOPs_M"], 2), n_seeds=len(seeds))
        rows.append(row)
        print(f"{name:32s} acc={row['acc_mean']*100:6.2f}+/-{row['acc_std']*100:4.2f}  "
              f"f1={row['f1_mean']*100:6.2f}  params={row['params']:>7,d}  "
              f"size={row['size_mb']:.3f}MB  FLOPs={row['FLOPs_M']:.1f}M")

    out_path = os.path.join(cfg["out_dir"], "ablation_components.json")
    with open(out_path, "w") as f:
        json.dump(rows, f, indent=2)
    print(f"Saved -> {out_path}")
    return rows


# ============================================================
# ABLATION STUDY 2: DEPTH AND WIDTH (capacity / efficiency frontier)
# "How deep" has two independent knobs in a CNN: DEPTH (how many ADA blocks
# are stacked at each resolution -- the `blocks` tuple) and WIDTH (how many
# channels each stage carries -- the `channels` tuple). We sweep both
# independently around the chosen base config so the trade-off curve is
# explicit, and flag every config against the 100K-parameter budget.
# ============================================================
DEPTH_VARIANTS = {
    "depth=(1,1,1,1) [default]": dict(blocks=(1, 1, 1, 1)),
    "depth=(2,1,1,1)":           dict(blocks=(2, 1, 1, 1)),
    "depth=(1,2,1,1)":           dict(blocks=(1, 2, 1, 1)),
    "depth=(1,1,2,1)":           dict(blocks=(1, 1, 2, 1)),
    "depth=(2,2,1,1)":           dict(blocks=(2, 2, 1, 1)),
    "depth=(1,2,2,1)":           dict(blocks=(1, 2, 2, 1)),
    "depth=(2,2,2,2)":           dict(blocks=(2, 2, 2, 2)),
}

WIDTH_VARIANTS = {
    "width=(8,16,24,32) [narrow]":  dict(channels=(8, 16, 24, 32)),
    "width=(12,24,32,40) [default]": dict(channels=(12, 24, 32, 40)),
    "width=(16,24,32,40)":          dict(channels=(16, 24, 32, 40)),
    "width=(16,32,40,48)":          dict(channels=(16, 32, 40, 48)),
    "width=(20,32,48,56) [wide]":   dict(channels=(20, 32, 48, 56)),
}


def run_depth_width_ablation(cfg, seeds=None, loaders=None, param_budget=100_000):
    """Ablation Study 2 -- depth sweep, then width sweep, around the default."""
    seeds = seeds or cfg["ablation_seeds"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if loaders is None:
        loaders = build_loaders(cfg)
    num_classes = len(loaders[3])

    def sweep(variants, title):
        print("\n" + "=" * 78)
        print(title)
        print("=" * 78)
        rows = []
        for name, override in variants.items():
            c = dict(cfg); c.update(override)
            comp = complexity(build_model(c, num_classes), c["img_size"])
            over_budget = comp["params"] > param_budget
            accs, f1s = [], []
            for sd in seeds:
                r = train_single(c, loaders, num_classes, sd, device)
                accs.append(r["test_acc"]); f1s.append(r["f1"])
            accs, f1s = np.array(accs), np.array(f1s)
            row = dict(variant=name,
                       acc_mean=float(accs.mean()), acc_std=float(accs.std(ddof=1) if len(accs) > 1 else 0.0),
                       f1_mean=float(f1s.mean()), f1_std=float(f1s.std(ddof=1) if len(f1s) > 1 else 0.0),
                       params=comp["params"], size_mb=round(comp["size_mb"], 4),
                       FLOPs_M=round(comp["FLOPs_M"], 2),
                       under_100k=not over_budget, n_seeds=len(seeds))
            rows.append(row)
            flag = "  [OVER 100K BUDGET]" if over_budget else ""
            print(f"{name:32s} acc={row['acc_mean']*100:6.2f}+/-{row['acc_std']*100:4.2f}  "
                  f"params={row['params']:>7,d}  FLOPs={row['FLOPs_M']:7.1f}M{flag}")
        return rows

    depth_rows = sweep(DEPTH_VARIANTS, "ABLATION STUDY 2a: Network DEPTH (blocks per stage)")
    width_rows = sweep(WIDTH_VARIANTS, "ABLATION STUDY 2b: Network WIDTH (channels per stage)")

    out_path = os.path.join(cfg["out_dir"], "ablation_depth_width.json")
    with open(out_path, "w") as f:
        json.dump(dict(depth=depth_rows, width=width_rows), f, indent=2)
    print(f"\nSaved -> {out_path}")

    # Pick the best accuracy/param trade-off that still respects the budget
    feasible = [r for r in (depth_rows + width_rows) if r["under_100k"]]
    best = max(feasible, key=lambda r: r["acc_mean"]) if feasible else None
    if best:
        print(f"\nBest under-100K config: {best['variant']} "
              f"(acc={best['acc_mean']*100:.2f}%, params={best['params']:,})")
    return depth_rows, width_rows


def run_full_ablation(cfg, seeds=None):
    """Convenience entry point: runs Study 1 then Study 2 on ONE shared data split."""
    loaders = build_loaders(cfg)
    comp_rows = run_component_ablation(cfg, seeds, loaders)
    depth_rows, width_rows = run_depth_width_ablation(cfg, seeds, loaders)
    return comp_rows, depth_rows, width_rows


# ------------------------------ OPTIONAL: SOTA pretrained baselines -----------
def train_torchvision_baseline(cfg, arch="mobilenet_v2", seed=0):
    """Same protocol/split for fair comparison (fix #3 & #7).
    arch in torchvision.models, e.g. resnet18, mobilenet_v2, efficientnet_b0,
    shufflenet_v2_x1_0, squeezenet1_1, densenet121 ..."""
    import torchvision.models as tvm
    device = "cuda" if torch.cuda.is_available() else "cpu"
    loaders = build_loaders(cfg); num_classes = len(loaders[3]); set_seed(seed)
    model = getattr(tvm, arch)(weights="DEFAULT")
    # swap classifier head to num_classes (handles common torchvision layouts)
    if hasattr(model, "fc"):
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif hasattr(model, "classifier"):
        if isinstance(model.classifier, nn.Sequential):
            last = model.classifier[-1]
            model.classifier[-1] = nn.Linear(last.in_features, num_classes)
        else:
            model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    model = model.to(device)
    crit = nn.CrossEntropyLoss()
    opt = AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sch = CosineAnnealingLR(opt, T_max=cfg["epochs"], eta_min=cfg["min_lr"])
    best, best_state = 0.0, None
    for ep in range(cfg["epochs"]):
        run_epoch(model, loaders[0], crit, opt, device, True)
        _, vacc = run_epoch(model, loaders[1], crit, opt, device, False)
        sch.step()
        if vacc > best: best, best_state = vacc, {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    P, Y, _ = predict(model, loaders[2], device)
    comp = complexity(model, cfg["img_size"])
    print(f"{arch}: acc={accuracy_score(Y,P)*100:.2f}  params={comp['params']:,}  FLOPs={comp['FLOPs_M']:.1f}M")
    np.savez(os.path.join(cfg["out_dir"], f"{arch}_preds.npz"), labels=Y, preds=P)
    return dict(arch=arch, acc=accuracy_score(Y, P), **comp, preds=P, labels=Y)


if __name__ == "__main__":
    # ----------------------------------------------------------------------
    # NO COMMAND-LINE ARGUMENTS. Edit these three lines directly, then just
    # run the cell / script. This avoids argparse entirely, which breaks in
    # Kaggle/Colab notebooks because the kernel launcher injects its own
    # sys.argv (e.g. "-f /tmp/xxx.json --HistoryManager.hist_file=...").
    # ----------------------------------------------------------------------
    MODE = "train"   # one of: "train", "ablation_components", "ablation_depth_width", "ablation_full", "baseline"
    ARCH = "mobilenet_v2"  # only used when MODE == "baseline"

    # CONFIG["data_dir"] above is a PLACEHOLDER Kaggle path. It will almost
    # certainly NOT match your environment. Find your real dataset folder
    # first (in a Kaggle notebook, run:  !ls /kaggle/input/  ) then set it
    # here, e.g.:
    #   CONFIG["data_dir"] = "/kaggle/input/<your-actual-folder-name>"
    # CONFIG["data_dir"] = "/kaggle/input/your-actual-folder-name"

    if MODE == "train":
        run_experiment(CONFIG)
    elif MODE == "ablation_components":
        run_component_ablation(CONFIG)
    elif MODE == "ablation_depth_width":
        run_depth_width_ablation(CONFIG)
    elif MODE == "ablation_full":
        run_full_ablation(CONFIG)
    elif MODE == "baseline":
        train_torchvision_baseline(CONFIG, arch=ARCH)
    else:
        raise ValueError(f"Unknown MODE: {MODE}")

Device: cuda
Found 4849 images across 6 classes: ['3_long_blade_rotor', '3_short_blade_rotor', 'Bird', 'Bird+mini-helicopter', 'RC_plane', 'drone']
  3_long_blade_rotor      : 799
  3_short_blade_rotor     : 800
  Bird                    : 800
  Bird+mini-helicopter    : 815
  RC_plane                : 800
  drone                   : 835
thop not available: No module named 'thop'

Complexity: {'params': 75005, 'size_mb': 0.2896, 'MACs_M': nan, 'FLOPs_M': nan}
Latency   : {'latency_ms_mean': 3.503, 'latency_ms_std': 0.5468, 'fps': 285.4714}

=== RAMD-Net seed 0 ===
    ep  1 val_acc=0.7538 best=0.7538
    ep 10 val_acc=0.9752 best=0.9752
    ep 20 val_acc=0.9794 best=0.9904
    ep 30 val_acc=0.9780 best=0.9904
    early stop @ epoch 31 (best val 0.9904)
    -> test_acc=0.9753 f1=0.9752

=== RAMD-Net seed 1 ===
    ep  1 val_acc=0.8693 best=0.8693
    ep 10 val_acc=0.9670 best=0.9670
    ep 20 val_acc=0.9766 best=0.9849
    ep 30 val_acc=0.9904 best=0.9904
    ep 40 val_acc=0.9849 best=0